[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/statistics/cox_proportional_hazards.ipynb)

# Cox Proportional Hazards: The Workhorse of Survival Analysis

Fit a Cox proportional hazards model to real recidivism data using Python's lifelines.
Learn the partial likelihood trick, hazard ratios, Schoenfeld diagnostics, and time-dependent covariates.

**Blog post:** [sesen.ai/blog/cox-proportional-hazards-recidivism](https://sesen.ai/blog/cox-proportional-hazards-recidivism)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from lifelines import CoxPHFitter, CoxTimeVaryingFitter, KaplanMeierFitter
from lifelines.statistics import proportional_hazard_test
from statsmodels.nonparametric.smoothers_lowess import lowess

np.random.seed(42)

## The Data: Recidivism After Prison

The Rossi recidivism dataset follows 432 male prisoners for one year after release.
The primary question: does receiving financial aid reduce the risk of rearrest?

In [ ]:
rossi_full = sm.datasets.get_rdataset("Rossi", "carData").data.copy()

# Encode categorical columns from R's factor format
rossi_full["fin"] = (rossi_full["fin"] == "yes").astype(int)
rossi_full["race"] = (rossi_full["race"] == "black").astype(int)
rossi_full["wexp"] = (rossi_full["wexp"] == "yes").astype(int)
rossi_full["mar"] = (rossi_full["mar"] == "married").astype(int)
rossi_full["paro"] = (rossi_full["paro"] == "yes").astype(int)

# Encode employment columns
emp_cols = [f"emp{i}" for i in range(1, 53)]
for c in emp_cols:
    rossi_full[c] = (rossi_full[c] == "yes").astype(float)

base_covs = ["fin", "age", "race", "wexp", "mar", "paro", "prio"]
df_base = rossi_full[["week", "arrest"] + base_covs].copy()

print(f"{len(df_base)} prisoners, {df_base['arrest'].sum()} rearrested")
df_base.head()

## Kaplan-Meier Survival Curve

A non-parametric estimate of the survival function before fitting any model.

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(df_base["week"], event_observed=df_base["arrest"], label="Overall")

fig, ax = plt.subplots(figsize=(9, 5))
kmf.plot_survival_function(ax=ax, ci_show=True, color="#2563EB", linewidth=2)
ax.fill_between(
    kmf.confidence_interval_survival_function_.index,
    kmf.confidence_interval_survival_function_.iloc[:, 0],
    kmf.confidence_interval_survival_function_.iloc[:, 1],
    alpha=0.15, color="#2563EB",
)
ax.set_xlabel("Weeks After Release", fontsize=12)
ax.set_ylabel("Proportion Not Rearrested", fontsize=12)
ax.set_title("Kaplan-Meier Survival Curve — Rossi Recidivism Data",
             fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11, loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Survival at 52 weeks: {kmf.predict(52):.3f}")

## Model 1: Cox PH with Baseline Covariates

Fit the Cox model with all 7 baseline covariates. The model estimates hazard ratios
without assuming any particular shape for the baseline hazard.

In [ ]:
cph = CoxPHFitter()
cph.fit(df_base, duration_col="week", event_col="arrest", show_progress=False)
cph.print_summary(columns=["coef", "exp(coef)", "se(coef)", "z", "p",
                            "exp(coef) lower 95%", "exp(coef) upper 95%"])
print(f"\nConcordance: {cph.concordance_index_:.4f}")

## Forest Plot: Hazard Ratios

Visualise the hazard ratios and their 95% confidence intervals.
Significant covariates (CI doesn't cross 1.0) are coloured.

In [ ]:
summary = cph.summary
hr = summary["exp(coef)"]
hr_lo = summary["exp(coef) lower 95%"]
hr_hi = summary["exp(coef) upper 95%"]
covnames = summary.index.tolist()

fig, ax = plt.subplots(figsize=(8, 5))
y_pos = np.arange(len(covnames))
ax.axvline(x=1, color="grey", linestyle="--", linewidth=1, alpha=0.7)

colors = ["#DC2626" if lo > 1 else "#2563EB" if hi < 1 else "#6B7280"
          for lo, hi in zip(hr_lo, hr_hi)]

for i, (name, h, lo, hi, col) in enumerate(
        zip(covnames, hr, hr_lo, hr_hi, colors)):
    ax.plot([lo, hi], [i, i], color=col, linewidth=2, solid_capstyle="round")
    ax.plot(h, i, "o", color=col, markersize=8, zorder=5)

ax.set_yticks(y_pos)
ax.set_yticklabels(
    [{"fin": "Financial Aid", "age": "Age", "race": "Race (Black)",
      "wexp": "Work Experience", "mar": "Married",
      "paro": "Parole", "prio": "Prior Convictions"}.get(c, c)
     for c in covnames], fontsize=11)
ax.set_xlabel("Hazard Ratio (95% CI)", fontsize=12)
ax.set_title("Cox PH Model — Hazard Ratios", fontsize=13, fontweight="bold")
ax.set_xlim(0, max(hr_hi) * 1.15)
ax.grid(True, axis="x", alpha=0.3)
ax.invert_yaxis()

for i, (h, lo, hi) in enumerate(zip(hr, hr_lo, hr_hi)):
    ax.annotate(f"{h:.2f} [{lo:.2f}, {hi:.2f}]",
                xy=(max(hr_hi) * 1.02, i), fontsize=9, va="center")

plt.tight_layout()
plt.show()

## Survival Curves by Financial Aid

Cox-adjusted survival curves comparing prisoners who received financial aid vs those who didn't,
with all other covariates held at their sample means.

In [ ]:
means = df_base[base_covs].mean()
indiv_0 = pd.DataFrame([means], columns=base_covs)
indiv_0["fin"] = 0
indiv_1 = pd.DataFrame([means], columns=base_covs)
indiv_1["fin"] = 1

fig, ax = plt.subplots(figsize=(9, 5))
sf0 = cph.predict_survival_function(indiv_0)
sf1 = cph.predict_survival_function(indiv_1)

# Bootstrap confidence intervals from parameter covariance
n_boot = 500
coef_mean = cph.params_.values
coef_cov = cph.variance_matrix_.values
boot_coefs = np.random.multivariate_normal(coef_mean, coef_cov, size=n_boot)
bch = cph.baseline_cumulative_hazard_
times = bch.index.values
X0 = indiv_0[cph.params_.index].values.flatten()
X1 = indiv_1[cph.params_.index].values.flatten()

surv_boot_0 = np.zeros((n_boot, len(times)))
surv_boot_1 = np.zeros((n_boot, len(times)))
for i, bc in enumerate(boot_coefs):
    surv_boot_0[i] = np.exp(-bch.values.flatten() * np.exp(np.dot(bc, X0)))
    surv_boot_1[i] = np.exp(-bch.values.flatten() * np.exp(np.dot(bc, X1)))

ci_lo_0, ci_hi_0 = np.percentile(surv_boot_0, [2.5, 97.5], axis=0)
ci_lo_1, ci_hi_1 = np.percentile(surv_boot_1, [2.5, 97.5], axis=0)

ax.step(times, sf0.values.flatten(), where="post", color="#DC2626",
        linewidth=2, label="No Financial Aid (fin=0)")
ax.fill_between(times, ci_lo_0, ci_hi_0, step="post", alpha=0.15, color="#DC2626")
ax.step(times, sf1.values.flatten(), where="post", color="#2563EB",
        linewidth=2, label="Financial Aid (fin=1)")
ax.fill_between(times, ci_lo_1, ci_hi_1, step="post", alpha=0.15, color="#2563EB")

ax.set_xlabel("Weeks After Release", fontsize=12)
ax.set_ylabel("Survival Probability", fontsize=12)
ax.set_title("Effect of Financial Aid on Survival", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11, loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Model 2: Time-Dependent Covariates (Employment)

Employment changes week to week, so we expand the data into start-stop format:
one row per person per week, with that week's employment status.

In [ ]:
rows = []
for idx, row in rossi_full.iterrows():
    person_id = idx
    event_week = int(row["week"])
    event = int(row["arrest"])
    for w in range(1, event_week + 1):
        emp_val = row[f"emp{w}"]
        if pd.isna(emp_val):
            continue
        r = {
            "id": person_id,
            "start": w - 1,
            "stop": w,
            "arrest": 1 if (w == event_week and event == 1) else 0,
            "employed": int(emp_val),
        }
        for c in base_covs:
            r[c] = row[c]
        rows.append(r)

df_td = pd.DataFrame(rows)
print(f"Expanded: {len(df_td)} person-week rows from {df_td['id'].nunique()} individuals")

ctv = CoxTimeVaryingFitter(penalizer=0.001)
ctv.fit(df_td, id_col="id", event_col="arrest",
        start_col="start", stop_col="stop", show_progress=False,
        formula="fin + age + race + wexp + mar + paro + prio + employed")
ctv.print_summary(columns=["coef", "exp(coef)", "se(coef)", "z", "p",
                            "exp(coef) lower 95%", "exp(coef) upper 95%"])

## Risk Trajectories

How predicted risk changes over time for three participants with different employment patterns.

In [ ]:
candidates = []
for idx in rossi_full.index:
    row = rossi_full.loc[idx]
    week = int(row["week"])
    if week < 30:
        continue
    emp_series = [row[f"emp{w}"] for w in range(1, week + 1)]
    emp_binary = [int(e) if not pd.isna(e) else 0 for e in emp_series]
    transitions = sum(1 for i in range(1, len(emp_binary))
                      if emp_binary[i] != emp_binary[i - 1])
    if transitions >= 2:
        candidates.append((idx, week, int(row["arrest"]), transitions, sum(emp_binary)))

candidates.sort(key=lambda x: -x[3])
selected_ids = []

if candidates:
    selected_ids.append(candidates[0][0])

never_emp = rossi_full[(rossi_full["arrest"] == 1) & (rossi_full["week"] >= 15)].index
for nid in never_emp:
    row = rossi_full.loc[nid]
    emp_any = any(row[f"emp{w}"] == 1 for w in range(1, int(row["week"]) + 1)
                  if not pd.isna(row[f"emp{w}"]))
    if not emp_any:
        selected_ids.append(nid)
        break

mostly_emp = rossi_full[(rossi_full["arrest"] == 0) & (rossi_full["week"] == 52)].index
for mid in mostly_emp:
    row = rossi_full.loc[mid]
    emp_sum = sum(1 for w in range(1, 53)
                  if not pd.isna(row[f"emp{w}"]) and row[f"emp{w}"] == 1)
    if emp_sum > 40:
        selected_ids.append(mid)
        break

fig, ax = plt.subplots(figsize=(9, 5))
line_colors = ["#DC2626", "#2563EB", "#059669"]
coefs_td = ctv.params_

for sel_id, col in zip(selected_ids, line_colors):
    person_data = df_td[df_td["id"] == sel_id].copy()
    if person_data.empty:
        continue
    row_orig = rossi_full.loc[sel_id]
    arrested = "Arrested" if row_orig["arrest"] == 1 else "Not Arrested"
    weeks = person_data["stop"].values
    cov_cols = coefs_td.index.tolist()
    X = person_data[cov_cols].values
    lp = X @ coefs_td.values
    rr = np.exp(lp)
    label = f"Person {sel_id} ({arrested}, emp={int(person_data['employed'].sum())}/{len(person_data)}wk)"
    ax.plot(weeks, rr, color=col, linewidth=1.8, label=label, alpha=0.9)
    emp_vals = person_data["employed"].values
    for i in range(len(weeks)):
        if emp_vals[i] == 1:
            ax.axvspan(weeks[i] - 1, weeks[i], alpha=0.05, color=col)

ax.set_xlabel("Weeks After Release", fontsize=12)
ax.set_ylabel("Relative Risk (exp(Xβ))", fontsize=12)
ax.set_title("Predicted Risk Trajectories — Time-Dependent Model",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="upper right")
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 53)
plt.tight_layout()
plt.show()

## Proportional Hazards Diagnostic

The PH assumption says hazard ratios stay constant over time.
We test this with Schoenfeld residuals on a reduced model (fin, age, prio).

In [ ]:
df_reduced = df_base[["week", "arrest", "fin", "age", "prio"]].copy()
cph_reduced = CoxPHFitter()
cph_reduced.fit(df_reduced, duration_col="week", event_col="arrest", show_progress=False)

ph_test = proportional_hazard_test(cph_reduced, df_reduced, time_transform="rank")
print(ph_test.summary)

scaled_schoenfeld = cph_reduced.compute_residuals(df_reduced, kind="scaled_schoenfeld")

# Map row indices back to actual event weeks
event_weeks = df_reduced.loc[scaled_schoenfeld.index, "week"].values

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
reduced_covs = ["fin", "age", "prio"]
pretty_names = {"fin": "Financial Aid", "age": "Age", "prio": "Prior Convictions"}

for i, cov in enumerate(reduced_covs):
    ax = axes[i]
    resids = scaled_schoenfeld[cov].values
    ax.scatter(event_weeks, resids, alpha=0.35, s=15, color="#6B7280", edgecolors="none")
    smoothed = lowess(resids, event_weeks, frac=0.6, return_sorted=True)
    ax.plot(smoothed[:, 0], smoothed[:, 1], color="#DC2626", linewidth=2, label="LOWESS")
    ax.axhline(y=cph_reduced.params_[cov], color="#2563EB", linestyle="--",
               linewidth=1, alpha=0.7, label=f"\u03b2 = {cph_reduced.params_[cov]:.3f}")
    ax.set_xlabel("Time (weeks)", fontsize=10)
    ax.set_ylabel("Scaled Schoenfeld Residual", fontsize=10)
    ax.set_title(pretty_names[cov], fontsize=11, fontweight="bold")
    ax.legend(fontsize=8, loc="best")
    ax.grid(True, alpha=0.3)

fig.suptitle("Proportional Hazards Diagnostic \u2014 Scaled Schoenfeld Residuals",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Cumulative Baseline Hazard

The Breslow estimator recovers the non-parametric baseline hazard after fitting.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bh = cph_reduced.baseline_cumulative_hazard_
ax.step(bh.index, bh.values.flatten(), where="post", color="#2563EB", linewidth=2)
ax.fill_between(bh.index, 0, bh.values.flatten(), step="post", alpha=0.1, color="#2563EB")
ax.set_xlabel("Weeks After Release", fontsize=12)
ax.set_ylabel("Cumulative Baseline Hazard", fontsize=12)
ax.set_title("Cumulative Baseline Hazard — Reduced Cox Model",
             fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercises

1. **Stratified Cox model.** Fit a stratified Cox model using `strata=['race']` in CoxPHFitter. How do the coefficients change when each racial group gets its own baseline hazard?

2. **Penalised Cox.** Increase the `penalizer` parameter in CoxPHFitter (e.g., 0.1, 1.0). Which covariates shrink towards zero first? Does concordance improve on held-out data?

3. **Log-rank test.** Use `lifelines.statistics.logrank_test` to formally test whether the survival curves for `fin=0` vs `fin=1` differ significantly. Compare the p-value to the Cox model's p-value for the `fin` coefficient.

4. **Bayesian comparison.** Refit this model in PyMC using a piecewise-exponential baseline hazard. Compare the posterior distributions for the `fin` effect to the frequentist confidence interval.

5. **Employment causality.** The time-dependent model shows employment has HR ≈ 0.35. Design a simple simulation that demonstrates how reverse causality (arrest prevents employment) can inflate the apparent protective effect of employment.